# Soreal — Knowledge Graph RAG Agent

**Story analysis and investigation** powered by SurrealDB + LangGraph + Ollama + LangSmith.

This notebook implements an agentic pipeline that:

1. **Ingests** unstructured text and chunks it
2. **Extracts** entities and relationships via a local LLM (Ollama)
3. **Verifies** triplets with a human-in-the-loop step
4. **Loads** verified knowledge into a SurrealDB graph (nodes + edges + vectors)
5. **Answers questions** via graph-traversal RAG — and **evolves** the graph during Q&A

### Prerequisites

| Dependency        | How to get it                                                                            |
| ----------------- | ---------------------------------------------------------------------------------------- |
| Python 3.10+      | Already installed                                                                        |
| `uv`              | `brew install uv`                                                                        |
| SurrealDB         | `brew install surrealdb/tap/surreal` then `surreal start --user root --pass root memory` |
| Ollama            | `brew install ollama` then `ollama pull llama3.1:8b`                                     |
| LangSmith API key | Free at https://smith.langchain.com — set in the config cell below                       |

Run the install cell once, restart the kernel, then continue.


In [ ]:
# ── Phase 0, Cell 1: Install dependencies ────────────────────────────────────
import shutil, subprocess, sys

packages = [
    "langchain-surrealdb",
    "langchain-ollama",
    "langchain-huggingface",
    "langgraph",
    "langsmith",
    "surrealdb",
    "sentence-transformers",
    "langchain-core",
    "langchain-community",
    "pandas",
]

if not shutil.which("uv"):
    raise RuntimeError("uv is not installed. Run: brew install uv")

cmd = ["uv", "pip", "install", "--python", sys.executable, "--upgrade", *packages]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("\n✅  Install complete. Restart the kernel before running the next cell.")

## Phase 0 — Configuration & Readiness


In [ ]:
# ── Phase 0, Cell 2: LangSmith + environment configuration ───────────────────
import os

# LangSmith observability — set your API key here or via env var
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "soreal-kg-agent")
# Uncomment and paste your key, or export LANGCHAIN_API_KEY before launching:
# os.environ["LANGCHAIN_API_KEY"] = "ls__..."

# SurrealDB
SURREAL_URL = os.getenv("SURREAL_URL", "ws://localhost:8000/rpc")
SURREAL_USERNAME = os.getenv("SURREAL_USERNAME", "root")
SURREAL_PASSWORD = os.getenv("SURREAL_PASSWORD", "root")
SURREAL_NAMESPACE = os.getenv("SURREAL_NAMESPACE", "soreal")
SURREAL_DATABASE = os.getenv("SURREAL_DATABASE", "kg")

# Ollama
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# Embeddings
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384

print("Configuration loaded.")
print(f"  SurrealDB:  {SURREAL_URL}")
print(f"  Ollama:     {OLLAMA_BASE_URL} / {OLLAMA_MODEL}")
print(f"  LangSmith:  project={os.environ.get('LANGCHAIN_PROJECT')}")

In [ ]:
# ── Phase 0, Cell 3: Readiness checks & connections ──────────────────────────
import socket, time, json
from urllib.parse import urlparse
import urllib.request

# --- SurrealDB check ---
def wait_for_surreal(url: str, timeout: float = 15.0) -> None:
    parsed = urlparse(url)
    host = parsed.hostname or "localhost"
    port = parsed.port or 8000
    deadline = time.time() + timeout
    last_err = None
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=2):
                return
        except OSError as e:
            last_err = e
            time.sleep(0.5)
    raise ConnectionError(
        f"SurrealDB not reachable at {url}. Start it in a separate terminal."
    ) from last_err

wait_for_surreal(SURREAL_URL)
print(f"✅  SurrealDB reachable at {SURREAL_URL}")

# --- Ollama check ---
def check_ollama(base_url: str, model: str) -> None:
    try:
        req = urllib.request.Request(f"{base_url}/api/tags")
        with urllib.request.urlopen(req, timeout=5) as resp:
            data = json.loads(resp.read())
            names = [m.get("name", "") for m in data.get("models", [])]
            if not any(model in n for n in names):
                print(f"⚠️  Model '{model}' not found. Available: {names}")
                print(f"   Run: ollama pull {model}")
            else:
                print(f"✅  Ollama ready with model {model}")
    except Exception as e:
        raise ConnectionError(
            f"Ollama not reachable at {base_url}. Start it with: ollama serve"
        ) from e

check_ollama(OLLAMA_BASE_URL, OLLAMA_MODEL)

# --- Connect to SurrealDB ---
from surrealdb import Surreal

conn = Surreal(SURREAL_URL)
conn.signin({"username": SURREAL_USERNAME, "password": SURREAL_PASSWORD})
conn.use(SURREAL_NAMESPACE, SURREAL_DATABASE)
print(f"✅  Connected: ns={SURREAL_NAMESPACE} db={SURREAL_DATABASE}")

# --- Embeddings ---
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✅  Embedding model loaded: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")

# --- LLM ---
from langchain_ollama import ChatOllama

llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
print(f"✅  LLM ready: {OLLAMA_MODEL}")

## Phase 1 — SurrealDB Schema & Agent Tools

Define the knowledge graph schema and the tools the agent will use.


In [ ]:
# ── Phase 1, Cell 4: Define SurrealDB schema ─────────────────────────────────

SCHEMA_STATEMENTS = [
    # Node tables
    "DEFINE TABLE IF NOT EXISTS character SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS label  ON character TYPE string",
    "DEFINE FIELD IF NOT EXISTS type   ON character TYPE option<string>",
    "DEFINE FIELD IF NOT EXISTS vector ON character TYPE option<array<float>>",

    "DEFINE TABLE IF NOT EXISTS object SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS label      ON object TYPE string",
    "DEFINE FIELD IF NOT EXISTS properties ON object TYPE option<array<string>>",
    "DEFINE FIELD IF NOT EXISTS vector     ON object TYPE option<array<float>>",

    "DEFINE TABLE IF NOT EXISTS action SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS label       ON action TYPE string",
    "DEFINE FIELD IF NOT EXISTS description ON action TYPE option<string>",

    "DEFINE TABLE IF NOT EXISTS thought SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS content ON thought TYPE string",

    "DEFINE TABLE IF NOT EXISTS chunk SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS content     ON chunk TYPE string",
    "DEFINE FIELD IF NOT EXISTS source      ON chunk TYPE option<string>",
    "DEFINE FIELD IF NOT EXISTS chunk_index ON chunk TYPE int",
    "DEFINE FIELD IF NOT EXISTS vector      ON chunk TYPE option<array<float>>",

    # Edge tables
    "DEFINE TABLE IF NOT EXISTS relates_to   TYPE RELATION SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS type         ON relates_to TYPE option<string>",
    "DEFINE FIELD IF NOT EXISTS source_chunk ON relates_to TYPE option<string>",

    "DEFINE TABLE IF NOT EXISTS performed    TYPE RELATION SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS source_chunk ON performed TYPE option<string>",

    "DEFINE TABLE IF NOT EXISTS has_thought  TYPE RELATION SCHEMAFULL",
    "DEFINE FIELD IF NOT EXISTS source_chunk ON has_thought TYPE option<string>",

    "DEFINE TABLE IF NOT EXISTS mentioned_in TYPE RELATION SCHEMAFULL",

    # Vector indexes (HNSW)
    "DEFINE INDEX IF NOT EXISTS idx_character_vec ON character FIELDS vector HNSW DIMENSION 384 DIST COSINE",
    "DEFINE INDEX IF NOT EXISTS idx_object_vec    ON object    FIELDS vector HNSW DIMENSION 384 DIST COSINE",
    "DEFINE INDEX IF NOT EXISTS idx_chunk_vec     ON chunk     FIELDS vector HNSW DIMENSION 384 DIST COSINE",
]

for stmt in SCHEMA_STATEMENTS:
    conn.query(stmt + ";")

print("✅  Schema defined. Verifying...")
info = conn.query("INFO FOR DB;")
print(json.dumps(info, indent=2)[:2000])

In [ ]:
# ── Phase 1, Cell 5: Define agent tools ───────────────────────────────────────
import hashlib
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------------------------------
# Tool 1: ingest_text — chunk text and store in SurrealDB
# ---------------------------------------------------------------------------
@tool
def ingest_text(text: str, source: str = "user_input") -> str:
    """Chunk unstructured text and store each chunk in SurrealDB with embeddings.
    Use this when the user provides new text to analyse."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_text(text)
    ids = []
    for i, chunk_text in enumerate(chunks):
        h = hashlib.md5(chunk_text.encode()).hexdigest()
        chunk_id = f"chunk:{h}"
        vec = embeddings.embed_query(chunk_text)
        conn.query(
            "CREATE type::thing($id) SET content=$content, source=$source, chunk_index=$idx, vector=$vec;",
            {"id": chunk_id, "content": chunk_text, "source": source, "idx": i, "vec": vec},
        )
        ids.append(chunk_id)
    return f"Ingested {len(ids)} chunks from '{source}': {ids}"


# ---------------------------------------------------------------------------
# Tool 2: extract_entities — LLM-based entity/relationship extraction
# ---------------------------------------------------------------------------
EXTRACTION_PROMPT = """You are an entity-extraction engine for story analysis.

Given the text below, extract entities and relationships matching this schema:

Entity types: character, object, action, thought
Relationship types:
  - relates_to (character→character, character→object, object→object) with a "type" label
  - performed (character→action)
  - has_thought (character→thought)

Return ONLY a valid JSON array. Each element:
{{
  "subject": "<entity label>",
  "subject_type": "<character|object|action|thought>",
  "predicate": "<relates_to|performed|has_thought>",
  "predicate_label": "<relationship label, e.g. partner_of, discovered>",
  "object": "<entity label>",
  "object_type": "<character|object|action|thought>"
}}

Text:
---
{text}
---

JSON array:"""


@tool
def extract_entities(chunk_ids: str) -> str:
    """Extract entities and relationships from chunks stored in SurrealDB.
    Input: comma-separated chunk IDs (e.g. 'chunk:abc,chunk:def').
    Returns JSON triplets for human verification."""
    import json as _json
    ids = [cid.strip() for cid in chunk_ids.split(",") if cid.strip()]
    all_triplets = []
    for cid in ids:
        rows = conn.query("SELECT content FROM type::thing($id);", {"id": cid})
        content = ""
        if rows and rows[0]:
            row = rows[0][0] if isinstance(rows[0], list) else rows[0]
            content = row.get("content", "")
        if not content:
            continue
        prompt = EXTRACTION_PROMPT.format(text=content)
        resp = llm.invoke(prompt)
        raw = resp.content.strip()
        try:
            start = raw.index("[")
            end = raw.rindex("]") + 1
            triplets = _json.loads(raw[start:end])
            for t in triplets:
                t["source_chunk"] = cid
            all_triplets.extend(triplets)
        except (ValueError, _json.JSONDecodeError):
            all_triplets.append({"error": f"Parse failed for {cid}", "raw": raw[:500]})
    return _json.dumps(all_triplets, indent=2)


# ---------------------------------------------------------------------------
# Tool 3: search_graph — vector entry-point + graph traversal
# ---------------------------------------------------------------------------
@tool
def search_graph(query: str) -> str:
    """Search the knowledge graph to answer questions about the story.
    Finds entry nodes via vector similarity, then traverses 1-2 hops."""
    import json as _json
    vec = embeddings.embed_query(query)

    # Find closest characters and objects
    entry_nodes = []
    for table in ["character", "object"]:
        results = conn.query(
            f"SELECT id, label, vector::similarity::cosine(vector, $vec) AS score "
            f"FROM {table} WHERE vector <|3,40|> $vec ORDER BY score DESC;",
            {"vec": vec},
        )
        if results and results[0]:
            items = results[0] if isinstance(results[0], list) else [results[0]]
            entry_nodes.extend(items)

    if not entry_nodes:
        # Fallback to chunk search
        chunk_res = conn.query(
            "SELECT id, content, source, vector::similarity::cosine(vector, $vec) AS score "
            "FROM chunk WHERE vector <|3,40|> $vec ORDER BY score DESC;",
            {"vec": vec},
        )
        if chunk_res and chunk_res[0]:
            items = chunk_res[0] if isinstance(chunk_res[0], list) else [chunk_res[0]]
            return _json.dumps({"source": "chunk_fallback", "results": items}, indent=2, default=str)
        return "No results found in the knowledge graph."

    # Graph traversal from entry nodes
    context_parts = []
    for node in entry_nodes:
        nid = node.get("id")
        if not nid:
            continue
        traversal = conn.query(
            "SELECT id, label, "
            "->relates_to->character.label AS related_characters, "
            "->relates_to->object.label AS related_objects, "
            "->performed->action.label AS actions_performed, "
            "->has_thought->thought.content AS thoughts, "
            "<-relates_to<-character.label AS related_from_characters "
            "FROM type::thing($nid);",
            {"nid": str(nid)},
        )
        context_parts.append({
            "node": node,
            "traversal": traversal[0] if traversal and traversal[0] else {},
        })

    return _json.dumps({"source": "graph_traversal", "results": context_parts}, indent=2, default=str)


# ---------------------------------------------------------------------------
# Tool 4: mutate_graph — insert/update nodes and edges
# ---------------------------------------------------------------------------
@tool
def mutate_graph(triplets_json: str) -> str:
    """Insert verified triplets into the SurrealDB knowledge graph.
    Input: JSON array of triplets with subject, subject_type, predicate,
    predicate_label, object, object_type, source_chunk (optional)."""
    import json as _json
    triplets = _json.loads(triplets_json)
    created_nodes = set()
    created_edges = 0

    def _node_id(label: str, ntype: str) -> str:
        h = hashlib.md5(f"{ntype}:{label}".lower().strip().encode()).hexdigest()
        return f"{ntype}:{h}"

    def _upsert_node(label: str, ntype: str) -> str:
        nid = _node_id(label, ntype)
        if nid in created_nodes:
            return nid
        vec = embeddings.embed_query(label) if ntype in ("character", "object") else None
        if ntype in ("character", "object"):
            conn.query(
                "CREATE type::thing($id) SET label=$label, vector=$vec;",
                {"id": nid, "label": label, "vec": vec},
            )
        elif ntype == "action":
            conn.query("CREATE type::thing($id) SET label=$label;", {"id": nid, "label": label})
        elif ntype == "thought":
            conn.query("CREATE type::thing($id) SET content=$label;", {"id": nid, "label": label})
        created_nodes.add(nid)
        return nid

    for t in triplets:
        subj_id = _upsert_node(t["subject"].strip(), t["subject_type"])
        obj_id = _upsert_node(t["object"].strip(), t["object_type"])
        predicate = t["predicate"]
        pred_label = t.get("predicate_label", "")
        src_chunk = t.get("source_chunk", "")
        conn.query(
            f"RELATE type::thing($from)->{predicate}->type::thing($to) SET type=$type, source_chunk=$src;",
            {"from": subj_id, "to": obj_id, "type": pred_label, "src": src_chunk},
        )
        created_edges += 1

    return f"Created {len(created_nodes)} nodes and {created_edges} edges."


# ---------------------------------------------------------------------------
# Tool 5: get_graph_summary
# ---------------------------------------------------------------------------
@tool
def get_graph_summary() -> str:
    """Get a summary of the current knowledge graph: node/edge counts and sample entities."""
    import json as _json
    summary = {}
    for table in ["character", "object", "action", "thought", "chunk"]:
        count_res = conn.query(f"SELECT count() AS c FROM {table} GROUP ALL;")
        c = 0
        if count_res and count_res[0]:
            row = count_res[0][0] if isinstance(count_res[0], list) else count_res[0]
            c = row.get("c", 0)
        label_field = "content" if table == "thought" else ("source" if table == "chunk" else "label")
        samples_res = conn.query(f"SELECT id, {label_field} FROM {table} LIMIT 5;")
        samples = samples_res[0] if samples_res and samples_res[0] else []
        summary[table] = {"count": c, "samples": samples}

    for edge in ["relates_to", "performed", "has_thought"]:
        count_res = conn.query(f"SELECT count() AS c FROM {edge} GROUP ALL;")
        c = 0
        if count_res and count_res[0]:
            row = count_res[0][0] if isinstance(count_res[0], list) else count_res[0]
            c = row.get("c", 0)
        summary[f"edge:{edge}"] = {"count": c}

    return _json.dumps(summary, indent=2, default=str)


ALL_TOOLS = [ingest_text, extract_entities, search_graph, mutate_graph, get_graph_summary]
print(f"✅  {len(ALL_TOOLS)} tools defined: {[t.name for t in ALL_TOOLS]}")

## Phase 2 — LangGraph Agent

A `StateGraph` with nodes for **ingestion**, **extraction**, **verification** (human-in-the-loop), **loading**, and **interactive Q&A**.


In [ ]:
# ── Phase 2, Cell 6: Agent state & graph definition ──────────────────────────
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    raw_triplets: str
    verified_triplets: str
    phase: str


# ── Graph nodes ──────────────────────────────────────────────────────────────

INGEST_TOOLS = [ingest_text, extract_entities]
QUERY_TOOLS = [search_graph, mutate_graph, get_graph_summary]

def ingest_node(state: AgentState) -> dict:
    """Process text ingestion and entity extraction via tool calls."""
    bound = llm.bind_tools(INGEST_TOOLS)
    system = SystemMessage(content=(
        "You are a knowledge-graph builder for story analysis. "
        "First call ingest_text with the user's text and source='user_input'. "
        "Then call extract_entities with ALL the returned chunk IDs (comma-separated). "
        "Return the extraction results verbatim."
    ))
    response = bound.invoke([system] + state["messages"])
    return {"messages": [response], "phase": "ingest"}


ingest_tools_executor = ToolNode(INGEST_TOOLS)


def should_continue_ingest(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "ingest_tools"
    # Check if extraction results are already in messages
    for msg in reversed(state["messages"]):
        content = str(getattr(msg, "content", ""))
        if '"subject"' in content and '"predicate"' in content:
            return "verify"
    return "ingest"  # loop back for another LLM call


def verify_node(state: AgentState) -> dict:
    """Present extracted triplets for human verification via interrupt."""
    # Find the extraction results in messages
    raw = ""
    for msg in reversed(state["messages"]):
        content = str(getattr(msg, "content", ""))
        if '"subject"' in content and '"predicate"' in content:
            # Extract the JSON array portion
            try:
                start = content.index("[")
                end = content.rindex("]") + 1
                raw = content[start:end]
            except ValueError:
                raw = content
            break

    review_msg = (
        "## Extracted Triplets for Review\n\n"
        f"```json\n{raw}\n```\n\n"
        "Edit the JSON if needed and paste the corrected version, "
        "or type **ok** to accept as-is."
    )
    user_response = interrupt(review_msg)

    verified = raw if user_response.strip().lower() == "ok" else user_response.strip()
    return {
        "raw_triplets": raw,
        "verified_triplets": verified,
        "messages": [AIMessage(content="Triplets verified. Loading into the knowledge graph...")],
        "phase": "verified",
    }


def load_node(state: AgentState) -> dict:
    """Load verified triplets into SurrealDB."""
    verified = state.get("verified_triplets", "")
    if not verified:
        return {"messages": [AIMessage(content="No verified triplets to load.")], "phase": "load"}
    result = mutate_graph.invoke({"triplets_json": verified})
    return {
        "messages": [AIMessage(content=f"Knowledge graph updated. {result}")],
        "phase": "loaded",
    }


def query_node(state: AgentState) -> dict:
    """ReAct-style Q&A agent with graph search and mutation tools."""
    bound = llm.bind_tools(QUERY_TOOLS)
    system = SystemMessage(content=(
        "You are a story-analysis assistant backed by a knowledge graph in SurrealDB. "
        "Use search_graph to find information before answering. "
        "If the user provides new facts, use mutate_graph to add them to the graph. "
        "Use get_graph_summary to see what's in the graph. "
        "Always ground your answers in the graph data returned by tools."
    ))
    response = bound.invoke([system] + state["messages"])
    return {"messages": [response], "phase": "query"}


query_tools_executor = ToolNode(QUERY_TOOLS)


def should_continue_query(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "query_tools"
    return END


# ── Build the graph ──────────────────────────────────────────────────────────

workflow = StateGraph(AgentState)

workflow.add_node("ingest", ingest_node)
workflow.add_node("ingest_tools", ingest_tools_executor)
workflow.add_node("verify", verify_node)
workflow.add_node("load", load_node)
workflow.add_node("query", query_node)
workflow.add_node("query_tools", query_tools_executor)

# Ingestion flow: START → ingest ↔ ingest_tools → verify → load → query ↔ query_tools → END
workflow.add_edge(START, "ingest")
workflow.add_conditional_edges("ingest", should_continue_ingest, {
    "ingest_tools": "ingest_tools",
    "verify": "verify",
    "ingest": "ingest",
})
workflow.add_edge("ingest_tools", "ingest")
workflow.add_edge("verify", "load")
workflow.add_edge("load", "query")
workflow.add_conditional_edges("query", should_continue_query, {
    "query_tools": "query_tools",
    END: END,
})
workflow.add_edge("query_tools", "query")

# Compile with checkpointer for persistent state
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("✅  LangGraph agent compiled")
print(f"   Nodes: {list(graph.nodes.keys())}")

In [ ]:
# Visualise the agent graph
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Phase 3 — Text Ingestion & Entity Extraction

Paste your story text below and run the cell. The agent will:

1. **Chunk** the text and store chunks in SurrealDB with embeddings
2. **Extract** entities and relationships (characters, objects, actions, thoughts)
3. **Pause** at the verification step for your review


In [ ]:
# ── Phase 3, Cell 8: Sample story text ────────────────────────────────────────
# Replace with your own text or load from a file.

story_text = """
Detective Marlow arrived at the old Thornfield manor just after midnight.
The front door was ajar, revealing a dimly lit hallway. On the marble floor
lay a brass pocket watch, its glass cracked, frozen at 11:47 PM.

Inside the study, Eleanor Thornfield sat motionless in a leather armchair.
She clutched a sealed envelope addressed to her late husband, Lord Alistair
Thornfield. "He told me he'd be safe," she whispered to no one in particular.

Marlow noticed scratch marks on the windowsill and a faint scent of
lavender—unusual for a house that had been closed for months. The family
butler, Mr. Graves, appeared silently from the kitchen carrying a silver
tray with two cups of tea. "Lord Thornfield always insisted on Ceylon,"
he remarked flatly.

A torn page from a journal was tucked behind the bookshelf. It read:
"I fear V. knows about the inheritance. If anything happens to me,
look beneath the sundial." The entry was dated three days before Lord
Thornfield's disappearance.

Marlow pocketed the journal page and turned to Graves. "Who else has
keys to this house?" Graves hesitated. "Only myself, Lady Thornfield,
and… Mr. Victor Hale. He is Lord Thornfield's business partner."
""".strip()

print(f"Story loaded: {len(story_text)} characters")

In [ ]:
# ── Phase 3, Cell 9: Run the ingestion pipeline ──────────────────────────────
# This streams the agent through ingest → extract → verify (interrupt).
# It will pause at the verify node and prompt you in the next cell.

import uuid

thread_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

events = []
for event in graph.stream(
    {"messages": [HumanMessage(content=f"Ingest and extract entities from this text:\n\n{story_text}")]},
    config=thread_config,
    stream_mode="values",
):
    events.append(event)
    # Print the last message from each step
    last_msg = event["messages"][-1]
    role = getattr(last_msg, "type", "unknown")
    content = str(getattr(last_msg, "content", ""))[:300]
    tool_calls = getattr(last_msg, "tool_calls", [])
    if tool_calls:
        print(f"[{role}] → tool calls: {[tc['name'] for tc in tool_calls]}")
    elif content:
        print(f"[{role}] {content[:200]}")

print("\n─── Pipeline paused at verification step ───")

## Phase 4 — Human-in-the-Loop Verification

The agent has paused after extraction. Inspect the extracted triplets below.

- Type **ok** to accept them as-is
- Or paste a corrected JSON array to override

Then run the resume cell to continue the pipeline (load → query).


In [ ]:
# ── Phase 4, Cell 10: Inspect pending state ───────────────────────────────────
# Show the current graph state (what the interrupt produced)

state = graph.get_state(thread_config)
print("Next node(s):", state.next)
print()

# The interrupt value contains the review message
for task in state.tasks:
    if hasattr(task, "interrupts"):
        for intr in task.interrupts:
            print(intr.value)

In [ ]:
# ── Phase 4, Cell 11: Approve / edit and resume ───────────────────────────────
# Set human_response to "ok" to accept, or paste corrected JSON.

human_response = "ok"  # <-- change this if you want to edit the triplets

# Resume the graph from the interrupt
for event in graph.stream(
    Command(resume=human_response),
    config=thread_config,
    stream_mode="values",
):
    last_msg = event["messages"][-1]
    role = getattr(last_msg, "type", "unknown")
    content = str(getattr(last_msg, "content", ""))
    tool_calls = getattr(last_msg, "tool_calls", [])
    if tool_calls:
        print(f"[{role}] → tool calls: {[tc['name'] for tc in tool_calls]}")
    elif content:
        print(f"[{role}] {content[:300]}")

print("\n─── Pipeline resumed: triplets loaded, agent ready for Q&A ───")

## Phase 5 — Knowledge Graph Verification

Query SurrealDB to confirm the knowledge graph was populated correctly.


In [ ]:
# ── Phase 5, Cell 12: Verify KG contents ──────────────────────────────────────
import pandas as pd

async def verify_kg():
    """Print node and edge counts + sample data."""
    tables = ["character", "object", "action", "thought", "chunk"]
    edge_tables = ["relates_to", "performed", "has_thought", "mentioned_in"]

    print("═══ Node Counts ═══")
    for t in tables:
        result = await conn.query(f"SELECT count() AS c FROM {t} GROUP ALL;")
        count = result[0]["result"][0]["c"] if result[0]["result"] else 0
        print(f"  {t:15s}: {count}")

    print("\n═══ Edge Counts ═══")
    for t in edge_tables:
        result = await conn.query(f"SELECT count() AS c FROM {t} GROUP ALL;")
        count = result[0]["result"][0]["c"] if result[0]["result"] else 0
        print(f"  {t:15s}: {count}")

    # Show characters
    chars = await conn.query("SELECT label, type FROM character;")
    if chars[0]["result"]:
        print("\n═══ Characters ═══")
        display(pd.DataFrame(chars[0]["result"]))

    # Show relationships
    rels = await conn.query(
        "SELECT in.label AS from_entity, type, out.label AS to_entity FROM relates_to;"
    )
    if rels[0]["result"]:
        print("\n═══ Relationships ═══")
        display(pd.DataFrame(rels[0]["result"]))

    # Show actions
    actions = await conn.query(
        "SELECT in.label AS who, out.label AS did_what FROM performed;"
    )
    if actions[0]["result"]:
        print("\n═══ Actions (character → action) ═══")
        display(pd.DataFrame(actions[0]["result"]))

await verify_kg()

In [ ]:
# ── Phase 5, Cell 13: Graph traversal example ─────────────────────────────────
# Demonstrate SurrealDB's native graph traversal capabilities

async def traverse_example():
    """Show multi-hop graph traversal from each character."""
    chars = await conn.query("SELECT label FROM character;")
    if not chars[0]["result"]:
        print("No characters found; run earlier cells first.")
        return

    for char in chars[0]["result"]:
        label = char["label"]
        print(f"\n{'═'*60}")
        print(f"Character: {label}")
        print(f"{'─'*60}")

        # 1-hop: what did this character do?
        actions = await conn.query(
            f"SELECT ->performed->action.label AS actions FROM character WHERE label = $label;",
            {"label": label}
        )
        if actions[0]["result"] and actions[0]["result"][0].get("actions"):
            print(f"  Actions: {actions[0]['result'][0]['actions']}")

        # 1-hop: who/what do they relate to?
        rels = await conn.query(
            f"SELECT ->relates_to.type AS rel_type, ->relates_to->?.label AS targets "
            f"FROM character WHERE label = $label;",
            {"label": label}
        )
        if rels[0]["result"]:
            for r in rels[0]["result"]:
                if r.get("targets"):
                    print(f"  Relates to: {r['targets']} (type: {r.get('rel_type', '?')})")

        # 1-hop: thoughts
        thoughts = await conn.query(
            f"SELECT ->has_thought->thought.content AS thoughts "
            f"FROM character WHERE label = $label;",
            {"label": label}
        )
        if thoughts[0]["result"] and thoughts[0]["result"][0].get("thoughts"):
            print(f"  Thoughts: {thoughts[0]['result'][0]['thoughts']}")

await traverse_example()

## Phase 6 — Graph-Traversal RAG Q&A

Ask investigative questions about the story. The agent will:

1. Find relevant entities via vector similarity
2. Traverse the knowledge graph for context
3. Generate a grounded answer

The agent can also **evolve** the graph — adding new facts discovered during Q&A.


In [ ]:
# ── Phase 6, Cell 14: Ask a question ──────────────────────────────────────────

def ask(question: str, config: dict):
    """Send a query-phase message to the agent and print the response."""
    print(f"❓  {question}\n")
    for event in graph.stream(
        {"messages": [HumanMessage(content=question)]},
        config=config,
        stream_mode="values",
    ):
        last_msg = event["messages"][-1]
        role = getattr(last_msg, "type", "unknown")
        content = str(getattr(last_msg, "content", ""))
        tool_calls = getattr(last_msg, "tool_calls", [])
        if tool_calls:
            print(f"  🔧 {[tc['name'] for tc in tool_calls]}")
        elif content and role == "ai":
            print(f"  💬 {content}\n")


# Sample investigative questions
ask("Who has keys to Thornfield manor?", thread_config)
ask("What is the connection between Victor Hale and Lord Thornfield?", thread_config)
ask("What clues did Detective Marlow find?", thread_config)

In [ ]:
# ── Phase 6, Cell 15: Evolving knowledge — add new facts via conversation ────
# The agent can add new information to the graph during Q&A.
# This demonstrates that the SurrealDB context EVOLVES during execution.

ask(
    "I just learned that Victor Hale owed Lord Thornfield £50,000 and they had "
    "a heated argument last Tuesday. Please add this to the knowledge graph.",
    thread_config
)

# Verify the new knowledge was added
ask("What do we now know about Victor Hale?", thread_config)

In [ ]:
# ── Phase 6, Cell 16: Verify graph evolution ──────────────────────────────────
# Re-run the KG verification to confirm new entities/edges were added

await verify_kg()

## Phase 7 — Observability & Persistent State

### LangSmith Tracing

Every LangGraph step, tool call, and LLM invocation is traced in LangSmith.
Open **[smith.langchain.com](https://smith.langchain.com)** → project **soreal-kg-agent** to inspect:

- Full agent execution traces
- Token usage per LLM call
- Tool invocation inputs/outputs
- Latency breakdowns per node

### Persistent State

The `MemorySaver` checkpointer preserves the full agent state between cells.
The agent can **resume from any checkpoint** — for example, re-running the verification step without re-ingesting text.


In [ ]:
# ── Phase 7, Cell 17: Observability info & persistent state demo ──────────────
import os

# LangSmith project URL
langsmith_project = os.environ.get("LANGCHAIN_PROJECT", "soreal-kg-agent")
print(f"🔗  LangSmith project: https://smith.langchain.com/o/default/projects/p/{langsmith_project}")
print(f"    Tracing enabled: {os.environ.get('LANGCHAIN_TRACING_V2', 'false')}")
print()

# Demonstrate persistent state: list all checkpoints for our thread
state = graph.get_state(thread_config)
print(f"═══ Persistent State (thread: {thread_config['configurable']['thread_id']}) ═══")
print(f"  Current phase: {state.values.get('phase', 'N/A')}")
print(f"  Message count: {len(state.values.get('messages', []))}")
print(f"  Has verified triplets: {bool(state.values.get('verified_triplets', ''))}")
print(f"  Next node(s): {state.next}")
print()

# Show that we can list state history (checkpoint trail)
print("═══ Checkpoint History ═══")
for i, snapshot in enumerate(graph.get_state_history(thread_config)):
    phase = snapshot.values.get("phase", "?")
    n_msgs = len(snapshot.values.get("messages", []))
    print(f"  [{i}] phase={phase}, messages={n_msgs}, next={snapshot.next}")
    if i >= 10:
        print("  ... (truncated)")
        break

---

## Demo Walkthrough

### Prerequisites

1. **SurrealDB** running locally: `surreal start --user root --pass root`
2. **Ollama** running with `llama3.1:8b` pulled: `ollama pull llama3.1:8b`
3. **LangSmith** API key set (optional but recommended for observability)

### Steps

1. Run **Phase 0** cells to install packages, configure environment, and verify connections
2. Run **Phase 1** to define the SurrealDB schema and agent tools
3. Run **Phase 2** to compile the LangGraph agent (inspect the visualized graph)
4. Run **Phase 3** to ingest the sample story text — the agent will chunk, embed, and extract entities
5. **Phase 4**: Review extracted triplets at the interrupt. Type `ok` or paste corrected JSON
6. Run **Phase 5** to verify the knowledge graph contents
7. Run **Phase 6** to ask investigative questions — watch the agent search and traverse the graph
8. Try the evolving knowledge cell — add new facts during Q&A and verify they persist
9. Run **Phase 7** to check LangSmith traces and persistent state

### Sample Questions

- "Who has keys to Thornfield manor?"
- "What is the connection between Victor Hale and Lord Thornfield?"
- "What clues did Detective Marlow find?"
- "What was written in the journal entry?"
- "Who might have a motive?"

### Judging Criteria Coverage

| Criterion                       | Weight | Coverage                                                          |
| ------------------------------- | ------ | ----------------------------------------------------------------- |
| **SurrealDB Structured Memory** | 30%    | Graph schema, vector indexes, node/edge tables, evolving context  |
| **LangGraph Agent Workflow**    | 20%    | StateGraph with 6 nodes, conditional edges, tool binding          |
| **Persistent Agent State**      | 20%    | MemorySaver checkpointer, resumable flows, checkpoint history     |
| **Practical Use Case**          | 20%    | Story analysis / investigation with entity extraction & graph Q&A |
| **Observability**               | 10%    | LangSmith tracing on all LLM calls, tool invocations, graph steps |
